In [0]:
from pyspark import pipelines as dp


@dp.table(
    name="silver.yellow_trips",
    comment="Validated Yellow Taxi trips",
    partition_cols=["year", "month", "day"]
)
@dp.expect_or_drop(
    "valid_yellow_trip",
    """
    tpep_pickup_datetime IS NOT NULL
    AND tpep_dropoff_datetime IS NOT NULL
    AND tpep_dropoff_datetime > tpep_pickup_datetime
    AND PULocationID IS NOT NULL
    AND DOLocationID IS NOT NULL
    AND PULocationID > 0
    AND DOLocationID > 0
    AND trip_distance >= 0
    """
)
def yellow_trips():
    return spark.readStream.table("bronze.yellow_trips_raw")


@dp.table(
    name="silver.green_trips",
    comment="Validated Green Taxi trips",
    partition_cols=["year", "month", "day"]
)
@dp.expect_or_drop(
    "valid_green_trip",
    """
    lpep_pickup_datetime IS NOT NULL
    AND lpep_dropoff_datetime IS NOT NULL
    AND lpep_dropoff_datetime > lpep_pickup_datetime
    AND PULocationID IS NOT NULL
    AND DOLocationID IS NOT NULL
    AND PULocationID > 0
    AND DOLocationID > 0
    AND trip_distance >= 0
    """
)
def green_trips():
    return (
        spark.readStream
        .table("bronze.green_trips_raw")
        .drop("ehail_fee")
    )